# Анализ результатов экспериментов

In [ ]:
import sys, os
from pathlib import Path


EXPERIMENT_DIRS = {
    "RU 1:1:1": "/path/to/ru_111",     
    "RU 3:3:1": "/path/to/ru_331",      
    "EN 1:1:1": "/path/to/en_111",     
    # "EN 3:3:1": "/path/to/en_331",   
}

MODEL_UTILS_DIR = os.path.abspath("model_utils")
if MODEL_UTILS_DIR not in sys.path:
    sys.path.insert(0, MODEL_UTILS_DIR)


OUTPUT_DIR = Path("analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(exist_ok=True)


MODEL_SHORT_NAMES = {
    "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli": "DeBERTa-v3",
    "BAAI/bge-m3": "BGE-M3",
    "deepvk/RuModernBERT-base": "ModernBERT",
}

TARGETS = ["relevance", "utilization", "adherence"]
F1_COLS = [f"{t}_f1" for t in TARGETS]
TEST_F1_COLS = [f"test_{t}_f1" for t in TARGETS]
P_COLS = [f"{t}_precision" for t in TARGETS]
R_COLS = [f"{t}_recall" for t in TARGETS]

print("Output dir:", OUTPUT_DIR.resolve())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

from results import load_results_from_dirs


frames = []
for exp_name, exp_dir in EXPERIMENT_DIRS.items():
    sweep_path = os.path.join(exp_dir, "sweeps")
    if not os.path.isdir(sweep_path):
        print(f"[!] Не найдена папка {sweep_path}, пропускаю {exp_name}")
        continue
    df = load_results_from_dirs(sweep_path)
    if df.empty:
        print(f"[!] Нет result.json в {sweep_path}, пропускаю {exp_name}")
        continue
    df["experiment"] = exp_name
    df["model_short"] = df["model_name"].map(MODEL_SHORT_NAMES).fillna(df["model_name"])
    frames.append(df)
    print(f"[+] {exp_name}: {len(df)} экспериментов загружено")

df_all = pd.concat(frames, ignore_index=True)
print(f"\nВсего загружено: {len(df_all)} экспериментов")
print(f"Эксперименты: {df_all['experiment'].unique().tolist()}")
print(f"Модели: {df_all['model_short'].unique().tolist()}")
print(f"Длины контекста: {sorted(df_all['max_length'].unique().tolist())}")
df_all.head()

## 2. Лучшая конфигурация для каждой модели (по val F1)

In [ ]:

idx_best = df_all.groupby(["experiment", "model_short"])["best_val_f1"].idxmax()
df_best = df_all.loc[idx_best].copy()

show_cols = [
    "experiment", "model_short", "max_length", "best_val_f1",
    *F1_COLS,
    *TEST_F1_COLS,
    "threshold_relevance", "threshold_utilization", "threshold_adherence",
]

show_cols = [c for c in show_cols if c in df_best.columns]

styled = (
    df_best[show_cols]
    .sort_values(["experiment", "best_val_f1"], ascending=[True, False])
    .reset_index(drop=True)
    .style
    .format("{:.4f}", subset=[c for c in show_cols if c not in ["experiment", "model_short", "max_length"]])
    .background_gradient(cmap="RdYlGn", subset=[c for c in F1_COLS + TEST_F1_COLS if c in show_cols], vmin=0, vmax=1)
)
styled

## 3. Полная сетка: Test F1 по всем (модель, длина, эксперимент)

In [ ]:


f1_source = TEST_F1_COLS if all(c in df_all.columns for c in TEST_F1_COLS) else F1_COLS
f1_label = "test" if f1_source == TEST_F1_COLS else "val"

for target in TARGETS:
    col = f"test_{target}_f1" if f"test_{target}_f1" in df_all.columns else f"{target}_f1"
    pivot = df_all.pivot_table(
        index=["model_short", "max_length"],
        columns="experiment",
        values=col,
        aggfunc="first",
    ).round(4)
    
    print(f"\n{'='*60}")
    print(f"  {target.upper()} F1 ({f1_label})")
    print(f"{'='*60}")
    display(pivot.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format("{:.4f}"))

    csv_path = OUTPUT_DIR / "tables" / f"grid_{target}_f1.csv"
    pivot.to_csv(csv_path)


full_cols = ["experiment", "model_short", "max_length"] + F1_COLS
if TEST_F1_COLS[0] in df_all.columns:
    full_cols += TEST_F1_COLS
full_cols = [c for c in full_cols if c in df_all.columns]

df_grid = (
    df_all[full_cols]
    .sort_values(["experiment", "model_short", "max_length"])
    .reset_index(drop=True)
)
df_grid.to_csv(OUTPUT_DIR / "tables" / "full_grid.csv", index=False)
print(f"\nСохранено в {OUTPUT_DIR / 'tables' / 'full_grid.csv'}")

## 4. Графики: F1 vs длина контекста (по эксперименту)

In [ ]:
MARKERS = ["o", "s", "^", "D", "v", "P", "X"]
COLORS = plt.cm.tab10.colors

def plot_f1_vs_length(df, experiment_name, use_test=True, save=True):
    """1x3 subplots: relevance/utilization/adherence F1 vs max_length, per model."""
    fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
    sub = df[df["experiment"] == experiment_name].copy()
    
    for ax, target in zip(axes, TARGETS):
        col = f"test_{target}_f1" if use_test and f"test_{target}_f1" in sub.columns else f"{target}_f1"
        
        for i, (model, g) in enumerate(sub.groupby("model_short")):
            g = g.sort_values("max_length")
            ax.plot(
                g["max_length"], g[col],
                marker=MARKERS[i % len(MARKERS)],
                color=COLORS[i % len(COLORS)],
                label=model, linewidth=2, markersize=7,
            )
        
        ax.set_xlabel("Max Length")
        ax.set_ylabel("F1")
        ax.set_title(f"{target.capitalize()} F1")
        ax.grid(True, alpha=0.3)
        ax.set_xscale("log", base=2)
        ax.set_xticks(sorted(sub["max_length"].unique()))
        ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
        ax.set_ylim(-0.02, 1.02)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.08), fontsize=10)
    fig.suptitle(f"F1 vs Context Length — {experiment_name}", y=1.14, fontsize=14, fontweight="bold")
    fig.tight_layout()
    
    if save:
        fname = OUTPUT_DIR / "plots" / f"f1_vs_length_{experiment_name.replace(' ', '_')}.png"
        fig.savefig(fname, bbox_inches="tight")
        print(f"Saved: {fname}")
    plt.show()


for exp in df_all["experiment"].unique():
    plot_f1_vs_length(df_all, exp)

## 5. лучшие модели test F1 по 3 таргетам

In [ ]:
def plot_best_models_bar(df, experiment_name, use_test=True, save=True):
    """Grouped bar chart: лучшая конфигурация каждой модели, 3 таргета."""
    sub = df[df["experiment"] == experiment_name]
    idx_best = sub.groupby("model_short")["best_val_f1"].idxmax()
    best = sub.loc[idx_best].sort_values("best_val_f1", ascending=False)
    
    cols = [f"test_{t}_f1" if use_test and f"test_{t}_f1" in best.columns else f"{t}_f1" for t in TARGETS]
    
    x = np.arange(len(best))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (col, target) in enumerate(zip(cols, TARGETS)):
        vals = best[col].values
        bars = ax.bar(x + i * width, vals, width, label=target.capitalize())
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8)
    
    labels = [f"{row['model_short']}\n(len={int(row['max_length'])})" for _, row in best.iterrows()]
    ax.set_xticks(x + width)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("F1")
    ax.set_ylim(0, 1.1)
    ax.set_title(f"Best Model Configs — {experiment_name}", fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    
    if save:
        fname = OUTPUT_DIR / "plots" / f"best_models_bar_{experiment_name.replace(' ', '_')}.png"
        fig.savefig(fname, bbox_inches="tight")
        print(f"Saved: {fname}")
    plt.show()

for exp in df_all["experiment"].unique():
    plot_best_models_bar(df_all, exp)

## 6. Веса лосса 1:1:1 vs 3:3:1

In [ ]:


WEIGHT_PAIRS = [
    ("RU 1:1:1", "RU 3:3:1"),
    ("EN 1:1:1", "EN 3:3:1"),
]

for exp_a, exp_b in WEIGHT_PAIRS:
    if exp_a not in df_all["experiment"].values or exp_b not in df_all["experiment"].values:
        print(f"[!] Пропускаю пару ({exp_a}, {exp_b}) — нет данных для одного из экспериментов")
        continue
    
    da = df_all[df_all["experiment"] == exp_a].set_index(["model_short", "max_length"])
    db = df_all[df_all["experiment"] == exp_b].set_index(["model_short", "max_length"])
    
    common_idx = da.index.intersection(db.index)
    if common_idx.empty:
        print(f"[!] Нет общих (модель, длина) для {exp_a} и {exp_b}")
        continue
    

    rows = []
    for target in TARGETS:
        col = f"test_{target}_f1" if f"test_{target}_f1" in da.columns else f"{target}_f1"
        for idx in common_idx:
            va = da.loc[idx, col] if idx in da.index else np.nan
            vb = db.loc[idx, col] if idx in db.index else np.nan
            if isinstance(va, pd.Series): va = va.iloc[0]
            if isinstance(vb, pd.Series): vb = vb.iloc[0]
            rows.append({
                "model": idx[0],
                "max_length": idx[1],
                "target": target,
                f"F1 ({exp_a})": va,
                f"F1 ({exp_b})": vb,
                "delta (3:3:1 - 1:1:1)": vb - va,
            })
    
    df_delta = pd.DataFrame(rows)
    print(f"\n{'='*70}")
    print(f"  Ablation: {exp_a} vs {exp_b}")
    print(f"{'='*70}")
    display(
        df_delta.style
        .format("{:.4f}", subset=[c for c in df_delta.columns if c not in ["model", "max_length", "target"]])
        .applymap(
            lambda v: "color: green" if isinstance(v, float) and v > 0.005 
                      else ("color: red" if isinstance(v, float) and v < -0.005 else ""),
            subset=["delta (3:3:1 - 1:1:1)"]
        )
    )
    
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot_delta = df_delta.pivot_table(
        index=["model", "max_length"], columns="target", values="delta (3:3:1 - 1:1:1)"
    )
    pivot_delta.plot(kind="bar", ax=ax, width=0.75)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Delta F1 (3:3:1 - 1:1:1)")
    ax.set_title(f"Effect of Loss Weights: {exp_a} vs {exp_b}", fontweight="bold")
    ax.legend(title="Target")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()
    
    fname = OUTPUT_DIR / "plots" / f"ablation_weights_{exp_a.replace(' ','_')}_vs_{exp_b.replace(' ','_')}.png"
    fig.savefig(fname, bbox_inches="tight")
    print(f"Saved: {fname}")
    plt.show()
    
    df_delta.to_csv(OUTPUT_DIR / "tables" / f"ablation_weights_{exp_a.replace(' ','_')}_vs_{exp_b.replace(' ','_')}.csv", index=False)

## 7. RU vs EN

In [ ]:

LANG_PAIRS = [
    ("RU 1:1:1", "EN 1:1:1"),
    ("RU 3:3:1", "EN 3:3:1"),
]

for exp_ru, exp_en in LANG_PAIRS:
    if exp_ru not in df_all["experiment"].values or exp_en not in df_all["experiment"].values:
        print(f"[!] Пропускаю пару ({exp_ru}, {exp_en}) — нет данных")
        continue
    
    d_ru = df_all[df_all["experiment"] == exp_ru].set_index(["model_short", "max_length"])
    d_en = df_all[df_all["experiment"] == exp_en].set_index(["model_short", "max_length"])
    
    common_idx = d_ru.index.intersection(d_en.index)
    if common_idx.empty:
        print(f"[!] Нет общих (модель, длина) для {exp_ru} и {exp_en}")
        continue
    
    rows = []
    for target in TARGETS:
        col = f"test_{target}_f1" if f"test_{target}_f1" in d_ru.columns else f"{target}_f1"
        for idx in common_idx:
            v_ru = d_ru.loc[idx, col]
            v_en = d_en.loc[idx, col]
            if isinstance(v_ru, pd.Series): v_ru = v_ru.iloc[0]
            if isinstance(v_en, pd.Series): v_en = v_en.iloc[0]
            rows.append({
                "model": idx[0],
                "max_length": idx[1],
                "target": target,
                f"F1 (RU)": v_ru,
                f"F1 (EN)": v_en,
                "delta (RU - EN)": v_ru - v_en,
            })
    
    df_lang = pd.DataFrame(rows)
    weights_label = exp_ru.split()[-1]
    
    print(f"\n{'='*70}")
    print(f"  RU vs EN (weights {weights_label})")
    print(f"{'='*70}")
    display(
        df_lang.style
        .format("{:.4f}", subset=[c for c in df_lang.columns if c not in ["model", "max_length", "target"]])
        .applymap(
            lambda v: "color: green" if isinstance(v, float) and v > 0.005
                      else ("color: red" if isinstance(v, float) and v < -0.005 else ""),
            subset=["delta (RU - EN)"]
        )
    )
    
    print("\nСредняя дельта (RU - EN) по таргетам:")
    print(df_lang.groupby("target")["delta (RU - EN)"].mean().round(4).to_string())
    
    fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
    for ax, target in zip(axes, TARGETS):
        sub = df_lang[df_lang["target"] == target].copy()
        sub["label"] = sub["model"] + "\n(len=" + sub["max_length"].astype(str) + ")"
        
        x = np.arange(len(sub))
        w = 0.35
        ax.bar(x - w/2, sub["F1 (RU)"], w, label="RU", color=COLORS[0])
        ax.bar(x + w/2, sub["F1 (EN)"], w, label="EN", color=COLORS[1])
        ax.set_xticks(x)
        ax.set_xticklabels(sub["label"], fontsize=7, rotation=45, ha="right")
        ax.set_title(f"{target.capitalize()} F1")
        ax.set_ylim(0, 1.1)
        ax.legend()
        ax.grid(axis="y", alpha=0.3)
    
    fig.suptitle(f"RU vs EN (weights {weights_label})", fontweight="bold", y=1.02)
    fig.tight_layout()
    fname = OUTPUT_DIR / "plots" / f"ru_vs_en_{weights_label.replace(':','')}.png"
    fig.savefig(fname, bbox_inches="tight")
    print(f"Saved: {fname}")
    plt.show()
    
    df_lang.to_csv(OUTPUT_DIR / "tables" / f"ru_vs_en_{weights_label.replace(':','')}.csv", index=False)

## 8. LLM-based baseline (Qwen2.5-72B) vs Classifier

In [ ]:

LLM_METRICS = {
    "covidqa": {
        "train": {"relevant_overlap": 0.7495, "utilized_overlap": 0.8507,
                   "relevant_miss": 1.289, "utilized_miss": 0.492,
                   "relevant_extra": 1.802, "utilized_extra": 0.866},
        "validation": {"relevant_overlap": 0.7768, "utilized_overlap": 0.8822,
                        "relevant_miss": 1.45, "utilized_miss": 0.5,
                        "relevant_extra": 1.425, "utilized_extra": 0.725},
        "test": {"relevant_overlap": 0.7716, "utilized_overlap": 0.8718,
                  "relevant_miss": 1.278, "utilized_miss": 0.417,
                  "relevant_extra": 1.972, "utilized_extra": 1.306},
    },
    "cuad": {
        "train": {"relevant_overlap": 0.7615, "utilized_overlap": 0.7782,
                   "relevant_miss": 0.397, "utilized_miss": 0.353,
                   "relevant_extra": 6.044, "utilized_extra": 2.691},
        "validation": {"relevant_overlap": 0.4591, "utilized_overlap": 0.7895,
                        "relevant_miss": 1.053, "utilized_miss": 0.316,
                        "relevant_extra": 7.105, "utilized_extra": 2.842},
        "test": {"relevant_overlap": 0.6946, "utilized_overlap": 0.8082,
                  "relevant_miss": 0.862, "utilized_miss": 0.828,
                  "relevant_extra": 6.172, "utilized_extra": 4.966},
    },
}

LLM_AGGREGATED = {
    "relevant_overlap_mean": 0.7022,
    "utilized_overlap_mean": 0.8301,
    "relevant_miss_mean": 1.0547,
    "utilized_miss_mean": 0.4842,
    "relevant_extra_mean": 4.0869,
    "utilized_extra_mean": 2.2326,
}

llm_rows = []
for dataset, splits in LLM_METRICS.items():
    for split, metrics in splits.items():
        llm_rows.append({"dataset": dataset, "split": split, **metrics})
df_llm = pd.DataFrame(llm_rows)

print("LLM-based evaluation (Qwen2.5-72B):")
display(df_llm.style.format("{:.4f}", subset=[c for c in df_llm.columns if c not in ["dataset", "split"]]))

print("\n" + "="*70)
print("  Сравнение: Classifier (лучшие) vs LLM (Qwen2.5-72B)")
print("="*70)
print("""
Примечание: метрики НЕ напрямую сравнимы:
  - Classifier: token-level F1 (precision/recall на уровне токенов)
  - LLM: sentence-level overlap ratio (доля правильно найденных ключей предложений)
  
Overlap ~= Recall на уровне предложений.
Для качественного сравнения нужно привести к единой шкале
(например, агрегировать classifier предсказания до уровня предложений).
""")

comparison_rows = []

comparison_rows.append({
    "method": "Qwen2.5-72B (LLM)",
    "relevance (sentence overlap)": LLM_AGGREGATED["relevant_overlap_mean"],
    "utilization (sentence overlap)": LLM_AGGREGATED["utilized_overlap_mean"],
    "relevance miss (avg)": LLM_AGGREGATED["relevant_miss_mean"],
    "utilization miss (avg)": LLM_AGGREGATED["utilized_miss_mean"],
})


if "RU 1:1:1" in df_all["experiment"].values:
    ru_best = df_all[df_all["experiment"] == "RU 1:1:1"]
    idx = ru_best.groupby("model_short")["best_val_f1"].idxmax()
    for i in idx:
        row = ru_best.loc[i]
        rel_col = "test_relevance_f1" if "test_relevance_f1" in row.index else "relevance_f1"
        util_col = "test_utilization_f1" if "test_utilization_f1" in row.index else "utilization_f1"
        comparison_rows.append({
            "method": f"{row['model_short']} (len={int(row['max_length'])})",
            "relevance (token F1)": row.get(rel_col, np.nan),
            "utilization (token F1)": row.get(util_col, np.nan),
        })

df_compare = pd.DataFrame(comparison_rows)
display(df_compare.style.format("{:.4f}", na_rep="—"))
df_compare.to_csv(OUTPUT_DIR / "tables" / "classifier_vs_llm.csv", index=False)

## 9. Анализ порогов

In [ ]:

thr_cols = ["threshold_relevance", "threshold_utilization", "threshold_adherence"]
available_thr = [c for c in thr_cols if c in df_all.columns]

if available_thr:
    print("Оптимальные пороги (подобраны на validation):\n")
    thr_show = ["experiment", "model_short", "max_length"] + available_thr
    display(
        df_all[thr_show]
        .sort_values(["experiment", "model_short", "max_length"])
        .reset_index(drop=True)
        .style.format("{:.2f}", subset=available_thr)
        .background_gradient(cmap="YlOrRd", subset=available_thr, vmin=0.1, vmax=0.7)
    )
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, col in zip(axes, available_thr):
        target_name = col.replace("threshold_", "")
        data_by_exp = [df_all[df_all["experiment"] == exp][col].dropna().values 
                       for exp in df_all["experiment"].unique()]
        bp = ax.boxplot(data_by_exp, labels=df_all["experiment"].unique(), patch_artist=True)
        for patch, color in zip(bp["boxes"], COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        ax.set_title(f"{target_name.capitalize()} threshold")
        ax.set_ylabel("Threshold")
        ax.grid(axis="y", alpha=0.3)
    
    fig.suptitle("Distribution of Optimal Thresholds", fontweight="bold", y=1.02)
    fig.tight_layout()
    fname = OUTPUT_DIR / "plots" / "threshold_boxplots.png"
    fig.savefig(fname, bbox_inches="tight")
    print(f"Saved: {fname}")
    plt.show()
    
    df_all[thr_show].to_csv(OUTPUT_DIR / "tables" / "thresholds.csv", index=False)
else:
    print("Пороги не найдены в данных (threshold_* колонки отсутствуют)")

## 10. Precision / Recall scatter 

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, target in zip(axes, TARGETS):
    p_col = f"test_{target}_precision" if f"test_{target}_precision" in df_all.columns else f"{target}_precision"
    r_col = f"test_{target}_recall" if f"test_{target}_recall" in df_all.columns else f"{target}_recall"
    
    if p_col not in df_all.columns or r_col not in df_all.columns:
        ax.set_title(f"{target} (no data)")
        continue
    
    for i, (exp, g) in enumerate(df_all.groupby("experiment")):
        for j, (model, mg) in enumerate(g.groupby("model_short")):
            ax.scatter(
                mg[r_col], mg[p_col],
                marker=MARKERS[j % len(MARKERS)],
                color=COLORS[i % len(COLORS)],
                s=80, alpha=0.7,
                label=f"{model} ({exp})" if target == TARGETS[0] else None,
            )
    
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"{target.capitalize()}")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.2)
    ax.grid(True, alpha=0.2)

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.15), fontsize=8)
fig.suptitle("Precision vs Recall (all experiments)", fontweight="bold", y=1.2)
fig.tight_layout()
fname = OUTPUT_DIR / "plots" / "precision_recall_scatter.png"
fig.savefig(fname, bbox_inches="tight")
print(f"Saved: {fname}")
plt.show()

## 11. LaTeX-таблицы

In [ ]:
def df_to_latex(df, caption, label, float_fmt="{:.4f}"):
    """Генерирует LaTeX-таблицу из DataFrame."""
    formatters = {}
    for col in df.columns:
        if df[col].dtype in [np.float64, np.float32]:
            formatters[col] = lambda x, fmt=float_fmt: fmt.format(x) if pd.notna(x) else "—"
    
    latex = df.to_latex(
        index=False,
        formatters=formatters,
        caption=caption,
        label=label,
        escape=True,
        column_format="l" + "c" * (len(df.columns) - 1),
    )
    return latex
if "RU 1:1:1" in df_all["experiment"].values:
    ru_data = df_all[df_all["experiment"].str.startswith("RU")]
    idx = ru_data.groupby(["experiment", "model_short"])["best_val_f1"].idxmax()
    best_ru = ru_data.loc[idx]
    
    cols_for_latex = ["experiment", "model_short", "max_length"]
    for t in TARGETS:
        test_col = f"test_{t}_f1"
        val_col = f"{t}_f1"
        cols_for_latex.append(test_col if test_col in best_ru.columns else val_col)
    cols_for_latex = [c for c in cols_for_latex if c in best_ru.columns]
    
    latex_table = best_ru[cols_for_latex].sort_values(["experiment", "best_val_f1"], ascending=[True, False])
    rename = {
        "experiment": "Experiment",
        "model_short": "Model",
        "max_length": "Max Len",
    }
    for t in TARGETS:
        rename[f"test_{t}_f1"] = f"{t.capitalize()} F1"
        rename[f"{t}_f1"] = f"{t.capitalize()} F1"
    
    latex_str = df_to_latex(
        latex_table.rename(columns=rename),
        caption="Best model configurations on Russian RAGBench (test F1)",
        label="tab:best_ru_results",
    )
    
    print(latex_str)
    
    with open(OUTPUT_DIR / "tables" / "best_ru_results.tex", "w", encoding="utf-8") as f:
        f.write(latex_str)
    print(f"Saved: {OUTPUT_DIR / 'tables' / 'best_ru_results.tex'}")

if len(comparison_rows) > 0:
    latex_str2 = df_to_latex(
        df_compare,
        caption="Comparison of classifier-based and LLM-based RAG evaluation",
        label="tab:classifier_vs_llm",
    )
    print("\n" + latex_str2)
    with open(OUTPUT_DIR / "tables" / "classifier_vs_llm.tex", "w", encoding="utf-8") as f:
        f.write(latex_str2)
    print(f"Saved: {OUTPUT_DIR / 'tables' / 'classifier_vs_llm.tex'}")

## 12. Сводка: все артефакты

In [ ]:

print("Сохранённые артефакты:\n")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.relative_to(OUTPUT_DIR)}  ({size_kb:.1f} KB)")

print(f"\nВсего экспериментов в анализе: {len(df_all)}")
print(f"Эксперименты: {df_all['experiment'].unique().tolist()}")
print(f"Модели: {df_all['model_short'].unique().tolist()}")
print(f"Длины: {sorted(df_all['max_length'].unique().tolist())}")